# Lab 1.3 - Generate Handwritten Digits with a VAE (Start)
**Day 1 - Session 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day1/lab1_3_vae/start/lab1_3_vae.ipynb)

**Goal:** Build and train a small variational autoencoder (VAE) on MNIST, then
sample its latent space to generate brand-new handwritten digits.

Look for the `# TODO` markers and fill them in. The `solution.ipynb` in this
folder has the full answer if you get stuck.

The dataset and epochs are capped so every cell finishes in a few minutes on a
Colab CPU runtime. No GPU needed.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q torch torchvision matplotlib

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 1. Load a slice of MNIST
We use 12,000 training images so training stays quick on CPU. Each image is a
28x28 grayscale handwritten digit.

In [ ]:
transform = transforms.ToTensor()
train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_ds = Subset(train_full, range(12000))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
print("Training images:", len(train_ds))

## 2. Define the VAE
The encoder maps an image to a mean and log-variance. The reparameterization
trick samples a latent vector `z` so gradients can still flow. The decoder
turns `z` back into a 28x28 image.

In [ ]:
LATENT_DIM = 16


class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(28 * 28, 256), nn.ReLU())
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 28 * 28), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc(x.view(-1, 28 * 28))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        # TODO: implement the reparameterization trick.
        # std = exp(0.5 * logvar); eps = randn_like(std); return mu + eps * std
        ...

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


model = VAE().to(device)
print(model)

## 3. The VAE loss
Two terms: reconstruction loss (the output should match the input) plus KL
divergence (keep the latent space close to a standard normal so we can sample
from it).

In [ ]:
def vae_loss(recon, x, mu, logvar):
    # TODO: return reconstruction loss + KL divergence.
    # recon_loss = F.binary_cross_entropy(recon, x.view(-1, 28*28), reduction="sum")
    # kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    ...

## 4. Train
Five epochs is enough to generate recognizable digits.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    running = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        # TODO: the training steps:
        #   1. optimizer.zero_grad()
        #   2. recon, mu, logvar = model(images)
        #   3. loss = vae_loss(recon, images, mu, logvar)
        #   4. loss.backward()
        #   5. optimizer.step()
        # Then add loss.item() to `running`.
        ...
    print(f"epoch {epoch + 1}: avg loss {running / len(train_ds):.2f}")

## 5. Generate new digits
Sample random latent vectors from a standard normal and decode them. These
digits were never in the training data.

In [ ]:
model.eval()
with torch.no_grad():
    # TODO: sample 16 latent vectors of size LATENT_DIM from a standard normal,
    # decode them, and reshape to (-1, 28, 28) on the CPU.
    # Hint: z = torch.randn(16, LATENT_DIM, device=device)
    samples = ...

fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for ax, img in zip(axes.flatten(), samples):
    ax.imshow(img, cmap="gray")
    ax.axis("off")
fig.suptitle("Digits generated from random latent vectors")
plt.show()

## Stretch goals
1. Interpolate between two latent vectors and decode each step to watch one
   digit morph into another.
2. Retrain with `LATENT_DIM = 2` and scatter the encoded training digits,
   coloring by their true label, to visualize the latent space.

In [ ]:
# TODO (stretch): interpolate between two latent points and plot the walk.

### Expected output
- The average loss drops each epoch.
- The 16 generated images look like plausible handwritten digits, a little
  blurry (VAEs favor smooth reconstructions over sharp detail).